# Trabalho Prático 2

Alunos:
- Bruno Figueiredo Lima (14562383)
- Guilherme Pascoale Godoy (14576277)

## Comandos

- `z` **rotaciona** a porta (abre/fecha)
- `i,j,k,l` **transladam** a raposa
- `e,r` **escalam** as duas taças liberatdores acima do móvel interno

## Estrutura do projeto

- `assets/models`: modelos 3D e texturas associadas.
- `assets/textures`: texturas de piso e skybox.
- `shaders`: arquivos de vertex e fragment shader.

Notebook dividido em:
1. bibliotecas e dependências
2. janela e shaders
3. funções auxiliares de carregamento
4. modelos e texturas da cena
5. buffers da GPU
6. controle da câmera
7. matrizes de transformação
8. renderização final

### Bibliotecas e dependências

Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from PIL import Image

from shader_s import Shader
from objects_manager import *
from notebook_loading_functions import *

### Inicialização da janela GLFW

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 720
largura = 1280

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Shaders do cenário e do skybox

#### Aqui a configuração foi modularizada, com a compilação ficando em arquivos próprios dos shaders.
Créditos: https://learnopengl.com

In [3]:
ourShader = Shader("shaders/vertex_shader.vs", "shaders/fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

# Compilando os shaders do SkyBox
skyboxShader = Shader("shaders/skybox.vs", "shaders/skybox.fs")
skybox_prog = skyboxShader.getProgram()

### Preparando dados para a GPU

Até aqui, compilamos nossos shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e precisam ser transmitidas para a GPU.

### Funções auxiliares para modelos, texturas e cubemap

A função abaixo carrega modelos a partir de arquivos WaveFront (.obj).

Para saber mais sobre o formato, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [4]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


#global vertices_list
#vertices_list = []    
#global textures_coord_list
#textures_coord_list = []
#global normals_list
#normals_list = []

# def load_model_from_file(filename):
#     """Loads a Wavefront OBJ file. """
#     objects = {}
#     vertices = []
#     normals = []
#     texture_coords = []
#     faces = []

#     material = None

#     # abre o arquivo obj para leitura
#     for line in open(filename, "r"): ## para cada linha do arquivo .obj
#         if line.startswith('#'): continue ## ignora comentarios
#         values = line.split() # quebra a linha por espaço
#         if not values: continue

#         ### recuperando vertices
#         if values[0] == 'v':
#             vertices.append(values[1:4])

#         ### recuperando normais
#         if values[0] == 'vn':
#             normals.append(values[1:4])

#         ### recuperando coordenadas de textura
#         elif values[0] == 'vt':
#             texture_coords.append(values[1:3])

#         ### recuperando faces 
#         elif values[0] in ('usemtl', 'usemat'):
#             material = values[1]
#         elif values[0] == 'f':
#             face = []
#             face_texture = []
#             face_normals = []
#             for v in values[1:]:
#                 w = v.split('/')
#                 face.append(int(w[0]))
#                 face_normals.append(int(w[2]))
#                 if len(w) >= 2 and len(w[1]) > 0:
#                     face_texture.append(int(w[1]))
#                 else:
#                     face_texture.append(0)

#             faces.append((face, face_texture, face_normals, material))

#     model = {}
#     model['vertices'] = vertices
#     model['texture'] = texture_coords
#     model['faces'] = faces
#     model['normals'] = normals ############ Novo!

#     return model

#def load_texture_from_file(texture_id, img_textura):
#    print(texture_id)
#    glBindTexture(GL_TEXTURE_2D, texture_id)
#    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
#    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
#    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
#    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
#    img = Image.open(img_textura)
#    img = img.convert("RGB")
#    img_width = img.size[0]
#    img_height = img.size[1]
#    image_data = img.tobytes("raw", "RGB", 0, -1)
#    #image_data = np.array(list(img.getdata()), np.uint8)
#    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

def load_ground_texture(path):
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    # Configura para repetir a imagem (Tiling)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    
    with Image.open(path) as img:
        img_data = img.convert("RGB").tobytes("raw", "RGB", 0, -1)
        glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img.width, img.height, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)
        glGenerateMipmap(GL_TEXTURE_2D)
    return texture_id

'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
# def circular_sliding_window_of_three(arr):
#     if len(arr) == 3:
#         return arr
#     circular_arr = arr + [arr[0]]
#     result = []
#     for i in range(len(circular_arr) - 2):
#         result.extend(circular_arr[i:i+3])
#     return result

# Carrega o cubo do Cube Map e as texturas das faces
def load_cubemap_from_files(faces_paths):
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_CUBE_MAP, texture_id)

    for i, path in enumerate(faces_paths):
        img = Image.open(path)
        img_data = img.convert("RGB").tobytes("raw", "RGB", 0, -1)
        # O enum de faces do Cubemap é sequencial (X+, X-, Y+, Y-, Z+, Z-)
        glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 0, GL_RGB, 
                     img.width, img.height, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)

    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)
    
    return texture_id

# Carrega as imagens de um único arquivo em formato de cruz
def load_cubemap_from_cross(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        w, h = img.size
        face_size = w // 4
        
        textureID = glGenTextures(1)
        glBindTexture(GL_TEXTURE_CUBE_MAP, textureID)
        
        # Garante leitura correta mesmo se face_size não for múltiplo de 4
        glPixelStorei(GL_UNPACK_ALIGNMENT, 1)

        # Offsets: Direita, Esquerda, Cima, Baixo, Frente, Trás
        offsets = [(2, 1), (0, 1), (1, 0), (1, 2), (1, 1), (3, 1)]
        
        for i, (col, row) in enumerate(offsets):
            left, top = col * face_size, row * face_size
            face_img = img.crop((left, top, left + face_size, top + face_size))
            
            # Corrige a inversão do eixo Y do Pillow para o OpenGL
            face_img = face_img.transpose(Image.FLIP_TOP_BOTTOM)
            
            img_data = face_img.tobytes("raw", "RGB", 0, -1)
            
            glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 0, GL_RGB, 
                         face_size, face_size, 0, GL_RGB, GL_UNSIGNED_BYTE, img_data)

    # Configurações de Filtro e Wrap (Essencial para não ver as "bordas" do cubo)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)
    
    return textureID


global numberTextures
numberTextures = 0

# def load_obj_and_texture(objFile, texturesList):
#     modelo = load_model_from_file(objFile)
    
#     ### inserindo vertices do modelo no vetor de vertices
#     verticeInicial = len(vertices_list)
#     print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    

#     #for face in modelo['faces']:
#     #    for vertice_id in face[0]:
#     #        vertices_list.append( modelo['vertices'][vertice_id-1] )
#     #    for texture_id in face[1]:
#     #        textures_coord_list.append( modelo['texture'][texture_id-1] )
#     #    for normal_id in face[2]:
#     #        normals_list.append( modelo['normals'][normal_id-1] )

#     faces_visited = []
#     for face in modelo['faces']:
#         if face[2] not in faces_visited:
#             faces_visited.append(face[2])
#         for vertice_id in circular_sliding_window_of_three(face[0]):
#             vertices_list.append(modelo['vertices'][vertice_id - 1])
#         for texture_id in circular_sliding_window_of_three(face[1]):
#             textures_coord_list.append(modelo['texture'][texture_id - 1])
#         for normal_id in circular_sliding_window_of_three(face[2]):
#             normals_list.append(modelo['normals'][normal_id - 1])
        
#     verticeFinal = len(vertices_list)
#     print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
#     ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
#     global numberTextures
#     for i in range(len(texturesList)):
#         load_texture_from_file(numberTextures,texturesList[i])
#         numberTextures += 1
    
#     return verticeInicial, verticeFinal - verticeInicial


# def load_object(folder):
#     # helper avancado: suporta objetos simples (1 textura) e multiplas texturas via MTL
#     global numberTextures
#     global vertices_list
#     global textures_coord_list
    
#     import os
#     import ntpath
#     from objects_manager import _find_first_file, parse_mtl, resolve_obj_and_texture, build_object
    
#     # Verifica se existe um arquivo .mtl na pasta
#     mtl_path = _find_first_file(folder, [".mtl"])
    
#     # Se não houver arquivo .mtl, usa o comportamento padrão (1 textura) antigo
#     if not mtl_path:
#         obj_path, texture_path = resolve_obj_and_texture(folder)
#         texture_id = numberTextures
#         vertice_inicial, quantos_vertices = load_obj_and_texture(obj_path, [texture_path])
#         return build_object(vertice_inicial, quantos_vertices, texture_id)

#     # ------------------ COMPORTAMENTO NOVO: MÚLTIPLOS MATERIAIS (.mtl) ------------------ #
#     print(f"Lendo modelo multi-material: {folder}")
#     obj_path = _find_first_file(folder, [".obj"])
#     modelo = load_model_from_file(obj_path)
#     materials_map = parse_mtl(mtl_path)
    
#     # Agrupa as faces por material
#     faces_por_material = {}
#     for face in modelo['faces']:
#         mat_name = face[3]  # face[2] contém o nome do material lido na tag "usemtl" do arquivo obj
#         if mat_name not in faces_por_material:
#             faces_por_material[mat_name] = []
#         faces_por_material[mat_name].append(face)
        
#     sub_objetos = []
    
#     # Carrega vértices e texturas SEPARADAMENTE para cada material
#     for mat_name, faces in faces_por_material.items():
#         tex_filename = materials_map.get(mat_name)
#         if not tex_filename:
#             continue  # Se esse material não aponta para nenhuma textura, apenas ignoramos
            
#         # O arquivo da textura esperado costuma ser apenas o nome no final, resolvemos caminhos complexos do windows se vierem no mtl
#         tex_nome = ntpath.basename(tex_filename)
#         tex_path = os.path.join(folder, tex_nome)
        
#         # Fallback inteligente: Se o arquivo citado no MTL por acaso não estiver ali com aquele mesmo nome,
#         # Nós usamos _find_first_file para tentar achar outra imagem pela extensão
#         if not os.path.exists(tex_path):
#             tex_path = _find_first_file(folder, [".jpg", ".png", ".jpeg", ".tif", ".tiff", ".bmp"])
        
#         if not tex_path:
#             continue # Nenhuma textura achada
            
#         # Guarda onde começam os vértices desta parte do modelo
#         verticeInicial = len(vertices_list)
        
#         # Insere apenas as faces correspondentes a esse material nas matrizes principais
#         for face in faces:
#             for vertice_id in circular_sliding_window_of_three(face[0]):
#                 vertices_list.append(modelo['vertices'][vertice_id - 1])
#             for texture_id in circular_sliding_window_of_three(face[1]):
#                 textures_coord_list.append(modelo['texture'][texture_id - 1])
#             for normals_id in circular_sliding_window_of_three(face[2]):
#                 textures_coord_list.append(modelo['normals'][normals_id - 1])
                
#         verticeFinal = len(vertices_list)
#         quantos_vertices = verticeFinal - verticeInicial
        
#         # Cria/carrega a textura e atrela o buffer
#         texture_id = numberTextures
#         load_texture_from_file(texture_id, tex_path)
#         numberTextures += 1
        
#         # Constroi o pequeno modelo que contem apenas os triângulos desse material!
#         parte = build_object(verticeInicial, quantos_vertices, texture_id, name=mat_name)
#         sub_objetos.append(parte)
        
#     return sub_objetos

In [5]:
# SkyBox

# Vértices do cubo
vertices_skybox = np.array([
    -1.0,  1.0, -1.0,  -1.0, -1.0, -1.0,   1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,   1.0,  1.0, -1.0,  -1.0,  1.0, -1.0,
    -1.0, -1.0,  1.0,  -1.0, -1.0, -1.0,  -1.0,  1.0, -1.0,
    -1.0,  1.0, -1.0,  -1.0,  1.0,  1.0,  -1.0, -1.0,  1.0,
     1.0, -1.0, -1.0,   1.0, -1.0,  1.0,   1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,   1.0,  1.0, -1.0,   1.0, -1.0, -1.0,
    -1.0, -1.0,  1.0,   1.0, -1.0,  1.0,   1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,  -1.0,  1.0,  1.0,  -1.0, -1.0,  1.0,
    -1.0,  1.0, -1.0,   1.0,  1.0, -1.0,   1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,  -1.0,  1.0,  1.0,  -1.0,  1.0, -1.0,
    -1.0, -1.0, -1.0,  -1.0, -1.0,  1.0,   1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,  -1.0, -1.0,  1.0,   1.0, -1.0,  1.0
], dtype='f4')

In [6]:
# Chão
# Vértices de um plano unitário (x, y, z, u, v)

glEnableVertexAttribArray(0)
glVertexAttribPointer(0, 3, GL_FLOAT, False, 5 * 4, ctypes.c_void_p(0))

glEnableVertexAttribArray(1)
glVertexAttribPointer(1, 2, GL_FLOAT, False, 5 * 4, ctypes.c_void_p(3 * 4))

# Plano unitário no eixo XZ (y=0) com repetição de textura de 100x
# Formato: x, y, z, u, v
vertices_grama = np.array([
    -0.5,  0.0, -0.5,    0.0,   0.0,  # Triângulo 1
     0.5,  0.0, -0.5,  100.0,   0.0,
     0.5,  0.0,  0.5,  100.0, 100.0,

    -0.5,  0.0, -0.5,    0.0,   0.0,  # Triângulo 2
     0.5,  0.0,  0.5,  100.0, 100.0,
    -0.5,  0.0,  0.5,    0.0, 100.0
], dtype='f4')

# Setup do VAO da grama
vao_grama = glGenVertexArrays(1)
vbo_grama = glGenBuffers(1)
glBindVertexArray(vao_grama)
glBindBuffer(GL_ARRAY_BUFFER, vbo_grama)
glBufferData(GL_ARRAY_BUFFER, vertices_grama.nbytes, vertices_grama, GL_STATIC_DRAW)
glEnableVertexAttribArray(0); glVertexAttribPointer(0, 3, GL_FLOAT, False, 5*4, None)
glEnableVertexAttribArray(1); glVertexAttribPointer(1, 2, GL_FLOAT, False, 5*4, ctypes.c_void_p(12))

### Carregamento dos objetos da cena

In [7]:
# carrega objetos (modelo e textura)
raposa = load_object('assets/models/raposa')
arvore = load_object('assets/models/arvores')
casa = load_object('assets/models/casa')
fogueira = load_object('assets/models/fogueira')
frango = load_object('assets/models/frango')
porta = load_object('assets/models/porta')
cama = load_object('assets/models/cama')
comoda = load_object('assets/models/comoda')
taca = load_object('assets/models/libertadores')
lampada = load_object('assets/models/lampada')

scene_objects = [
    ("raposa", raposa, (-90, 1, 0, 0, 6, -1.1, -16, 0.05, 0.05, 0.05)),
    ("arvore", arvore, (0.0, 0, 1, 0,  15, -1, -20,  5.0, 5.0, 5.0)),
    ("casa", casa, (270, 0, 1, 0,  0, -1, -30,  1.5, 1.5, 1.5)),
    ("fogueira",fogueira, (0.0, 0, 1, 0, -6, -1, -16, 0.05, 0.05, 0.05)),
    ("frango",frango, (90, 0, 0, 1, 3.3, -0.6, -14, 0.005, 0.005, -0.005)),
    ("porta", porta, (270, 0, 1, 0, 1.4, 1.8, -22.7, 1.5, 1.5, 1.5)),
    ("cama", cama, (0, 0, 1, 0, 4, 2.03, -31, 2, 2, 2)),
    ("comoda",comoda, (90, 0, 1, 0, -6.7, 2.03, -25, 0.02, 0.02, 0.02)),
    ("taca", taca, (0, 0, 1, 0, -5.6, 3.98, -24.2, 0.2, 0.2, 0.2)),
    ("taca", taca, (0, 0, 1, 0, -5.6, 3.98, -25.8, 0.2, 0.2, 0.2)),
    ("lampada", lampada, (0, 0, 1, 0, 1.8, 3.15, -36, 1.5, 1.5, 1.5)),
]

Lendo modelo multi-material: assets/models/raposa
0
Lendo modelo multi-material: assets/models/arvores
1
2
Lendo modelo multi-material: assets/models/casa
3
4
5
6
7
Lendo modelo multi-material: assets/models/fogueira
8
Lendo modelo multi-material: assets/models/frango
9
Lendo modelo multi-material: assets/models/porta
10
Lendo modelo multi-material: assets/models/cama
11
12
Processando modelo assets/models/comoda/eb_dresser_01.obj. Vertice inicial: 738252
Processando modelo assets/models/comoda/eb_dresser_01.obj. Vertice final: 742842
13
Lendo modelo multi-material: assets/models/libertadores
14
15
16
17
Lendo modelo multi-material: assets/models/lampada
18
19
20
21
22
23


### Adicionando fontes de luz


In [8]:
class Light:
    def __init__(self, position, ambient, diffuse, specular):
        self.position = glm.vec3(position)
        self.ambient = glm.vec3(ambient)
        self.diffuse = glm.vec3(diffuse)
        self.specular = glm.vec3(specular)

# Lista de luzes da cena
luzes = [
    Light(
        position=[1.8, 3.15, -36],
        ambient=[0.0, 1.0, 1.0],
        diffuse=[0.0, 0.0, 0.0],
        specular=[0.0, 0.0, 0.0]
    )
]

### Buffers da GPU

Para enviar nossos dados da CPU para a GPU, precisamos requisitar três slots (buffers): um para os vértices, um para as texturas e um para as normais.

In [9]:
buffer_VBO = glGenBuffers(3)

### Upload das posições dos vértices

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [10]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Carregar a textura
cubemap_texture = load_cubemap_from_cross("assets/textures/skybox/ceu_skybox.jpg")
#cubemap_texture = load_cubemap_from_files(["assets/models/caixa/caixa.jpg", "assets/models/caixa/caixa.jpg", "assets/models/caixa/caixa.jpg", "assets/models/caixa/caixa.jpg", "assets/models/caixa/caixa.jpg", "assets/models/caixa/caixa.jpg"])

#Carregar as texturas de chão
tex_grama = load_ground_texture("assets/textures/piso/grass.jpg")

# Criar VAO/VBO específico para Skybox
skybox_VAO = glGenVertexArrays(1)
skybox_VBO = glGenBuffers(1)

glBindVertexArray(skybox_VAO)
glBindBuffer(GL_ARRAY_BUFFER, skybox_VBO)
glBufferData(GL_ARRAY_BUFFER, vertices_skybox.nbytes, vertices_skybox, GL_STATIC_DRAW)

glEnableVertexAttribArray(0)
glVertexAttribPointer(0, 3, GL_FLOAT, False, 3 * 4, ctypes.c_void_p(0))
glBindVertexArray(0) # Fecha o VAO do Skybox com segurança

# --- NOVIDADE: Crie um VAO para a cena para não conflitar ---
vao_cena = glGenVertexArrays(1)
glBindVertexArray(vao_cena) # Ativa o VAO da cena

# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)

stride = vertices.strides[0]
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, ctypes.c_void_p(0))

glBindVertexArray(0) # Fecha o VAO da cena

### Upload das coordenadas de textura

In [11]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list

glBindVertexArray(vao_cena)

# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

glBindVertexArray(0)

### Upload das normais

In [12]:
normals = np.zeros(len(normals_list), [("position", np.float32, 3)]) # três coordenadas
normals['position'] = normals_list


# Upload coordenadas normals de cada vertice
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])

glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)
stride = normals.strides[0]
offset = ctypes.c_void_p(0)
loc_normals_coord = glGetAttribLocation(program, "normals")
glEnableVertexAttribArray(loc_normals_coord)
glVertexAttribPointer(loc_normals_coord, 3, GL_FLOAT, False, stride, offset)

glBindVertexArray(0)

### Upload dos atributos das luzes

In [13]:
num_luzes = len(luzes)
glUniform1i(glGetUniformLocation(program, "numActiveLights"), num_luzes)

# 3. Passa os dados de cada luz para o array do shader
for i, luz in enumerate(luzes):
    # Monta o nome base pra acessar a variável no shader
    base = f"lights[{i}]."
    # glm.value_ptr() pra acessar os valores
    glUniform3fv(glGetUniformLocation(program, base + "position"), 1, glm.value_ptr(luz.position))
    glUniform3fv(glGetUniformLocation(program, base + "ambient"),  1, glm.value_ptr(luz.ambient))
    glUniform3fv(glGetUniformLocation(program, base + "diffuse"),  1, glm.value_ptr(luz.diffuse))
    glUniform3fv(glGetUniformLocation(program, base + "specular"), 1, glm.value_ptr(luz.specular))

### Controle da câmera e interação

* Usei as teclas A, S, D e W para movimentação no espaço tridimensional
* Usei a posição do mouse para "direcionar" a câmera

In [14]:
#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

# camera
cameraPos   = glm.vec3(0.0, 8.0, 3.0)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

# Abrir/fechar porta
# 1 -> abre
# -1 -> fecha
# 0 -> parada
rotate_porta = -1

# escala da taca libertadores
escala_taca = 0.2

# pos da raposa
raposa_x = 6
raposa_z = -16

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode, rotate_porta, escala_taca, raposa_x, raposa_z

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    # limita pelo piso
    cameraPos.y = max(-0.8, cameraPos.y)

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode
        
    if key == glfw.KEY_Z and action == glfw.PRESS:
        rotate_porta = -rotate_porta
    
    if key == glfw.KEY_E and (action == glfw.PRESS or action == glfw.REPEAT):
        escala_taca = max(0.1, escala_taca-0.01)
    
    if key == glfw.KEY_R and (action == glfw.PRESS or action == glfw.REPEAT):
        escala_taca = min(0.3, escala_taca+0.01)

    # controla raposa
    if key == glfw.KEY_I and (action == glfw.PRESS or action == glfw.REPEAT):
        raposa_z -= 0.2

    if key == glfw.KEY_K and (action == glfw.PRESS or action == glfw.REPEAT):
        raposa_z += 0.2
    
    if key == glfw.KEY_J and (action == glfw.PRESS or action == glfw.REPEAT):
        raposa_x -= 0.2
    
    if key == glfw.KEY_L and (action == glfw.PRESS or action == glfw.REPEAT):
        raposa_x += 0.2



def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)


# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)


# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [15]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    translation = np.array([
        [1.0, 0.0, 0.0, t_x],
        [0.0, 1.0, 0.0, t_y],
        [0.0, 0.0, 1.0, t_z],
        [0.0, 0.0, 0.0, 1.0],
    ], dtype=np.float32)
    
    if angle != 0:
        c = math.cos(angle)
        s = math.sin(angle)
        rotation = np.identity(4, dtype=np.float32)

        if r_x != 0:
            rotation_x = np.array([
                [1.0, 0.0, 0.0, 0.0],
                [0.0, c, -s, 0.0],
                [0.0, s, c, 0.0],
                [0.0, 0.0, 0.0, 1.0],
            ], dtype=np.float32)
            rotation = rotation @ rotation_x

        if r_y != 0:
            rotation_y = np.array([
                [c, 0.0, s, 0.0],
                [0.0, 1.0, 0.0, 0.0],
                [-s, 0.0, c, 0.0],
                [0.0, 0.0, 0.0, 1.0],
            ], dtype=np.float32)
            rotation = rotation @ rotation_y

        if r_z != 0:
            rotation_z = np.array([
                [c, -s, 0.0, 0.0],
                [s, c, 0.0, 0.0],
                [0.0, 0.0, 1.0, 0.0],
                [0.0, 0.0, 0.0, 1.0],
            ], dtype=np.float32)
            rotation = rotation @ rotation_z
    else:
        rotation = np.identity(4, dtype=np.float32)
    
    scale = np.array([
        [s_x, 0.0, 0.0, 0.0],
        [0.0, s_y, 0.0, 0.0],
        [0.0, 0.0, s_z, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ], dtype=np.float32)
    
    matrix_transform = translation @ rotation @ scale
    
    return matrix_transform.astype(np.float32)

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Exibição da janela

In [16]:
glfw.show_window(window)

### Loop principal de renderização

In [ ]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

while not glfw.window_should_close(window):

    ourShader.use()

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)

    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection) 
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)


    # PORTA (ROTAÇÃO) ----------------------------

    porta_idxs = [idx for idx, scene in enumerate(scene_objects) if scene[0]=="porta"]

    if rotate_porta == 1: 
        p = list(scene_objects[porta_idxs[0]][2])

        if p[0] > 130:
            p[0] -= 0.5
        else:
            p[0] = 130

        scene_objects[porta_idxs[0]] = (scene_objects[porta_idxs[0]][0], scene_objects[porta_idxs[0]][1], tuple(p))


    elif rotate_porta == -1:
        p = list(scene_objects[porta_idxs[0]][2])

        if p[0] < 270:
            p[0] += 0.5
        else:
            p[0] = 270

        scene_objects[porta_idxs[0]] = (scene_objects[porta_idxs[0]][0], scene_objects[porta_idxs[0]][1], tuple(p))

    # LIBERTADORES (ESCALA) ----------------------------

    taca_idxs = [idx for idx, scene in enumerate(scene_objects) if scene[0]=="taca"]

    for i in taca_idxs:
        t = list(scene_objects[i][2])
        t[-1] = t[-2] = t[-3] = escala_taca
        scene_objects[i] = (scene_objects[i][0], scene_objects[i][1], tuple(t))


    # RAPOSA (TRANSLAÇÃO) ------------------------------------
    raposa_idx = [idx for idx, scene in enumerate(scene_objects) if scene[0]=="raposa"][0]

    t = list(scene_objects[raposa_idx][2])
    t[-6] = raposa_x
    t[-4] = raposa_z
    scene_objects[raposa_idx] = (scene_objects[raposa_idx][0], scene_objects[raposa_idx][1], tuple(t))


    glBindVertexArray(vao_cena)
    for _, obj, params in scene_objects:
        mat_model = model(*params)
        draw_object(obj, mat_model, program)

    # Desenha piso externo
    glBindTexture(GL_TEXTURE_2D, tex_grama)
    mat_grama = model(0, 0,1,0, 0, -1.0, 0, 1000, 1, 1000) # chão "infinito"
    glUniformMatrix4fv(glGetUniformLocation(program, "model"), 1, GL_TRUE, mat_grama)
    glBindVertexArray(vao_grama)
    glDrawArrays(GL_TRIANGLES, 0, 6)

    glDepthFunc(GL_LEQUAL) # Muda para aceitar o fundo infinito (z=1.0)
    
    skyboxShader.use()

    glUniformMatrix4fv(glGetUniformLocation(skybox_prog, "view"), 1, GL_TRUE, mat_view)
    glUniformMatrix4fv(glGetUniformLocation(skybox_prog, "projection"), 1, GL_TRUE, mat_projection)

    glActiveTexture(GL_TEXTURE0)
    glBindTexture(GL_TEXTURE_CUBE_MAP, cubemap_texture)
    glUniform1i(glGetUniformLocation(skybox_prog, "skybox"), 0)
    
    glBindVertexArray(skybox_VAO)
    glDrawArrays(GL_TRIANGLES, 0, 36)
    
    glDepthFunc(GL_LESS)

    glfw.swap_buffers(window)

glfw.terminate()